In [ ]:
import os, sys
sys.path.append('..')
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-5"

## 1. Uso Básico del System Prompt

In [ ]:
def ask_with_system(user_msg: str, system_msg: str) -> str:
    resp = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=system_msg,
        messages=[{"role": "user", "content": user_msg}]
    )
    return resp.content[0].text

# Sin system prompt
resp_sin = ask_with_system(
    "¿Qué es la fotosíntesis?",
    ""
)
print("Sin system prompt:")
print(resp_sin[:300])

In [ ]:
# Con system prompt que define rol y audiencia
system_educador = """
Eres un educador de primaria que explica conceptos científicos 
a niños de 8 años. Usa palabras simples, analogías con cosas 
cotidianas y máximo 3 oraciones.
"""

resp_con = ask_with_system("¿Qué es la fotosíntesis?", system_educador)
print("Con system prompt (educador niños):")
print(resp_con)

## 2. Diferentes Roles y Personas

In [ ]:
personas = {
    "Chef": """Eres un chef profesional con 20 años de experiencia en cocina 
    mediterránea. Respondes siempre con entusiasmo culinario y terminas 
    tus respuestas con una recomendación de maridaje.""",
    
    "Abogado": """Eres un abogado corporativo. Das respuestas precisas y 
    estructuradas, siempre mencionando que esto no es asesoría legal formal 
    y que deben consultar un profesional para su caso específico.""",
    
    "Poeta": """Eres un poeta modernista. Todas tus respuestas tienen un 
    tono poético y usas metáforas creativas. Jamás das respuestas directas, 
    siempre las envuelves en lenguaje figurado.""",
}

pregunta = "¿Qué hago si me duele la cabeza?"

for persona, system in personas.items():
    resp = ask_with_system(pregunta, system)
    print(f"\n[{persona}]")
    print(resp[:250])
    print("-" * 60)

## 3. System Prompts para Control de Formato

In [ ]:
system_json = """
Responde SIEMPRE en formato JSON válido con esta estructura exacta:
{
  "respuesta": "texto de la respuesta",
  "confianza": "alta|media|baja",
  "temas_relacionados": ["tema1", "tema2"]
}
No incluyas ningún texto fuera del JSON.
"""

resp = ask_with_system("¿Cuál es el planeta más grande del sistema solar?", system_json)
print(resp)

# Parsear el JSON
import json
data = json.loads(resp)
print("\nJSON parseado correctamente:")
print(f"  Respuesta: {data['respuesta']}")
print(f"  Confianza: {data['confianza']}")


## 4. Ejercicio: Asistente Personalizado

**Tarea**: Crea un system prompt para un asistente de servicio al cliente de una tienda de tecnología llamada **TechMundo**. El asistente debe:
1. Identificarse como "Techo" (el asistente de TechMundo)
2. Responder siempre en español
3. Ser amable y profesional
4. Si no sabe algo, ofrecer transferir con un agente humano

In [ ]:
system_techmundo = """
Eres Techo, el asistente virtual de TechMundo, una tienda lider en tecnologia.
- Presentate siempre como "Techo de TechMundo" al inicio de la conversacion.
- Responde siempre en español, de forma amable y profesional.
- Ayuda con preguntas sobre productos, garantias, envios y devoluciones.
- Si no sabes la respuesta, di: "Para ese tema te transfiero con uno de nuestros 
  agentes especializados. Te parece bien?"
- Finaliza cada respuesta con "Puedo ayudarte con algo mas?"
"""

# Prueba tu system prompt
preguntas_prueba = [
    "Hola, que ofertas tienen hoy?",
    "Mi laptop se danio, que hago?",
    "Cual es la politica de devoluciones?",
]

for pregunta in preguntas_prueba:
    resp = ask_with_system(pregunta, system_techmundo)
    print(f"\nCliente: {pregunta}")
    print(f"Techo: {resp[:300]}")
    print("-" * 60)
